# Bug detection: character TF-IDF + Logistic Regression

A simple NLP baseline for classifying Python functions as **CLEAN (0)** or **BUGGY (1)**.
Run the cells from top to bottom. The CSV is already prepared; this notebook does not execute the source code in it.

We compare three character n-gram ranges using validation data, then evaluate the selected model once on test data.
Select the project's `.venv` Python kernel in your notebook editor.

In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from sklearn.model_selection import train_test_split

project_dir = next(
    folder for folder in [Path.cwd(), Path.cwd().parent, Path.cwd() / "nlp-code-assistant"]
    if (folder / "data/processed/bug_detection_dataset.csv").exists()
)
df = pd.read_csv(
    project_dir / "data/processed/bug_detection_dataset.csv",
    dtype={"source_id": str},
    keep_default_na=False,
)
assert df["label"].isin([0, 1]).all()
assert df["source_id"].str.strip().ne("").all()
assert df["code"].str.strip().ne("").all()

print("Rows:", len(df))
print("Clean:", (df["label"] == 0).sum())
print("Buggy:", (df["label"] == 1).sum())
print("\nMutation type distribution:")
print(df["mutation_type"].value_counts().to_string())

Rows: 14743
Clean: 10000
Buggy: 4743

Mutation type distribution:
mutation_type
clean                  10000
comparison_operator     1956
arithmetic_operator     1838
boolean_operator         949


## 1. Split by original function

Split **source IDs**, rather than individual rows, into approximately 70% training, 15% validation, and 15% test.
This keeps the clean and mutated versions of a function together. Otherwise, nearly identical functions could appear in both training and evaluation, making the scores misleading.
The percentages apply to source IDs; the row proportions can differ slightly.

In [2]:
source_ids = sorted(df["source_id"].unique())
train_ids, held_out_ids = train_test_split(source_ids, test_size=0.30, random_state=42)
val_ids, test_ids = train_test_split(held_out_ids, test_size=0.50, random_state=42)

train_df = df[df["source_id"].isin(train_ids)].copy()
val_df = df[df["source_id"].isin(val_ids)].copy()
test_df = df[df["source_id"].isin(test_ids)].copy()

assert set(train_ids).isdisjoint(val_ids)
assert set(train_ids).isdisjoint(test_ids)
assert set(val_ids).isdisjoint(test_ids)
assert len(train_df) + len(val_df) + len(test_df) == len(df)

for name, split in [("Train", train_df), ("Validation", val_df), ("Test", test_df)]:
    assert set(split["label"]) == {0, 1}
    print(f"{name}: {len(split)} rows, {split['source_id'].nunique()} source IDs")
print("Source ID overlap: 0 for all split pairs")

y_train = train_df["label"]
y_val = val_df["label"]
y_test = test_df["label"]

Train: 10315 rows, 7000 source IDs
Validation: 2206 rows, 1500 source IDs
Test: 2222 rows, 1500 source IDs
Source ID overlap: 0 for all split pairs


## 2. Initial baseline and small validation experiment

TF-IDF converts each function into a sparse vector of weighted text features: patterns common in a function receive more weight, while patterns common across many functions receive less weight.

Character n-grams capture short syntax patterns around operators such as `==`, `!=`, `>=`, `<=`, `+`, `-`, and `*`, as well as `and` and `or`. We preserve case because Python is case-sensitive. They describe local text patterns, not the meaning or correctness of the program.

Logistic Regression learns weights for these features to predict CLEAN versus BUGGY. `class_weight="balanced"` accounts for the unequal class counts.

Start with `(3, 5)`, then try `(2, 4)` and `(3, 6)`. All settings except the n-gram range stay fixed. This notebook uses the requested vectorizer defaults, including ordinary term frequency (`sublinear_tf=False`), so its results can differ from the earlier script's sublinear-TF experiment.

In [ ]:
results = []
best_f1 = -1

for ngram_range in [(3, 5), (2, 4), (3, 6)]:
    print(f"\nTraining n-grams {ngram_range} ...", flush=True)
    vectorizer = TfidfVectorizer(
        analyzer="char",
        ngram_range=ngram_range,
        max_features=30000,
        lowercase=False,
    )
    X_train = vectorizer.fit_transform(train_df["code"])
    X_val = vectorizer.transform(val_df["code"])

    model = LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        random_state=42,
    )
    model.fit(X_train, y_train)
    val_predictions = model.predict(X_val)
    report = classification_report(
        y_val, val_predictions, labels=[0, 1],
        target_names=["CLEAN", "BUGGY"], output_dict=True, zero_division=0,
    )
    buggy = report["BUGGY"]
    results.append({
        "ngram_range": ngram_range,
        "precision": buggy["precision"],
        "recall": buggy["recall"],
        "f1": buggy["f1-score"],
    })
    print("Validation Buggy F1:", round(buggy["f1-score"], 4))

    if ngram_range == (3, 5):
        print("\nINITIAL BASELINE: VALIDATION")
        print(classification_report(
            y_val, val_predictions, labels=[0, 1],
            target_names=["CLEAN", "BUGGY"], digits=4, zero_division=0,
        ))
        print("Accuracy:", accuracy_score(y_val, val_predictions))
        print("Confusion matrix [[TN, FP], [FN, TP]]:")
        print(confusion_matrix(y_val, val_predictions, labels=[0, 1]))


    if buggy["f1-score"] > best_f1:
        best_f1 = buggy["f1-score"]
        best_ngram_range = ngram_range
        best_vectorizer = vectorizer
        best_model = model
        best_val_predictions = val_predictions

comparison = pd.DataFrame(results).sort_values("ngram_range")
print("\nVALIDATION COMPARISON (BUGGY CLASS)")
print(comparison.to_string(index=False, float_format=lambda value: f"{value:.4f}"))
print("\nSelected n-gram range:", best_ngram_range)
print("\nSELECTED MODEL: VALIDATION")
print(classification_report(
    y_val, best_val_predictions, labels=[0, 1],
    target_names=["CLEAN", "BUGGY"], digits=4, zero_division=0,
))
print("Accuracy:", accuracy_score(y_val, best_val_predictions))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_val, best_val_predictions, labels=[0, 1]))


Training n-grams (3, 5) ...
Validation Buggy F1: 0.5748

INITIAL BASELINE: VALIDATION
              precision    recall  f1-score   support

       CLEAN     0.8132    0.7053    0.7554      1500
       BUGGY     0.5116    0.6558    0.5748       706

    accuracy                         0.6895      2206
   macro avg     0.6624    0.6806    0.6651      2206
weighted avg     0.7167    0.6895    0.6976      2206

Accuracy: 0.6894832275611967
Confusion matrix [[TN, FP], [FN, TP]]:
[[1058  442]
 [ 243  463]]

Training n-grams (2, 4) ...
Validation Buggy F1: 0.5779

Training n-grams (3, 6) ...
Validation Buggy F1: 0.5659

VALIDATION COMPARISON (BUGGY CLASS)
ngram_range  precision  recall     f1
     (2, 4)     0.5210  0.6487 0.5779
     (3, 5)     0.5116  0.6558 0.5748
     (3, 6)     0.5028  0.6473 0.5659

Selected n-gram range: (2, 4)

SELECTED MODEL: VALIDATION
              precision    recall  f1-score   support

       CLEAN     0.8131    0.7193    0.7634      1500
       BUGGY     0.5

## 3. Final test evaluation

The configuration is now fixed. Transform test code with the selected training-fitted vectorizer and evaluate the retained model once. Do not use these results to choose another configuration.

In [4]:
X_test = best_vectorizer.transform(test_df["code"])
test_predictions = best_model.predict(X_test)

test_report = classification_report(
    y_test, test_predictions, labels=[0, 1],
    target_names=["CLEAN", "BUGGY"], output_dict=True, zero_division=0,
)
print(classification_report(
    y_test, test_predictions, labels=[0, 1],
    target_names=["CLEAN", "BUGGY"], digits=4, zero_division=0,
))
print("Confusion matrix [[TN, FP], [FN, TP]]:")
print(confusion_matrix(y_test, test_predictions, labels=[0, 1]))
print("Accuracy:", round(accuracy_score(y_test, test_predictions), 4))
print("Buggy precision:", round(test_report["BUGGY"]["precision"], 4))
print("Buggy recall:", round(test_report["BUGGY"]["recall"], 4))
print("Buggy F1:", round(test_report["BUGGY"]["f1-score"], 4))

              precision    recall  f1-score   support

       CLEAN     0.8206    0.6953    0.7528      1500
       BUGGY     0.5195    0.6842    0.5906       722

    accuracy                         0.6917      2222
   macro avg     0.6700    0.6898    0.6717      2222
weighted avg     0.7228    0.6917    0.7001      2222

Confusion matrix [[TN, FP], [FN, TP]]:
[[1043  457]
 [ 228  494]]
Accuracy: 0.6917
Buggy precision: 0.5195
Buggy recall: 0.6842
Buggy F1: 0.5906


## 4. Detection by mutation type

Among the truly BUGGY test rows, count how many predictions are BUGGY. For each mutation type, recall is `detected / total`.

In [5]:
test_results = test_df.copy()
test_results["predicted_label"] = test_predictions
buggy_results = test_results[test_results["label"] == 1]

mutation_results = buggy_results.groupby("mutation_type").agg(
    total=("label", "size"),
    detected=("predicted_label", "sum"),
)
mutation_results["recall"] = mutation_results["detected"] / mutation_results["total"]
print(mutation_results.to_string(float_format=lambda value: f"{value:.4f}"))

                     total  detected  recall
mutation_type                               
arithmetic_operator    269       185  0.6877
boolean_operator       142        78  0.5493
comparison_operator    311       231  0.7428


## 5. Inspect mistakes

A **false positive** is clean code predicted as buggy. A **false negative** is buggy code predicted as clean. Print the first five of each, including their complete code, for the report.

In [6]:
false_positives = test_results[
    (test_results["label"] == 0) & (test_results["predicted_label"] == 1)
]
false_negatives = test_results[
    (test_results["label"] == 1) & (test_results["predicted_label"] == 0)
]
label_names = {0: "CLEAN", 1: "BUGGY"}

for title, errors in [("FALSE POSITIVES", false_positives), ("FALSE NEGATIVES", false_negatives)]:
    print(f"\n{title}: {len(errors)} total; showing up to 5")
    for _, row in errors.head(5).iterrows():
        print("-" * 60)
        print("source_id:", row["source_id"])
        print("mutation_type:", row["mutation_type"])
        print("true_label:", label_names[row["label"]])
        print("predicted_label:", label_names[row["predicted_label"]])
        print("CODE:\n" + row["code"])


FALSE POSITIVES: 457 total; showing up to 5
------------------------------------------------------------
source_id: 000018
mutation_type: clean
true_label: CLEAN
predicted_label: BUGGY
CODE:
def getcosmo(cosmology):
    

    defaultcosmologies = {'dragons': cg.DRAGONS(), 'wmap1': cg.WMAP1_Mill(),
                          'wmap3': cg.WMAP3_ML(), 'wmap5': cg.WMAP5_mean(),
                          'wmap7': cg.WMAP7_ML(), 'wmap9': cg.WMAP9_ML(),
                          'wmap1_lss': cg.WMAP1_2dF_mean(),
                          'wmap3_mean': cg.WMAP3_mean(),
                          'wmap5_ml': cg.WMAP5_ML(),
                          'wmap5_lss': cg.WMAP5_BAO_SN_mean(),
                          'wmap7_lss': cg.WMAP7_BAO_H0_mean(),
                          'planck13': cg.Planck_2013(),
                          'planck15': cg.Planck_2015()}

    if isinstance(cosmology, dict):
        # User providing their own variables
        cosmo = cosmology
        if 'A_scaling' not in cosm

## 6. Short explanation for the presentation

1. **TF-IDF** represents code as numeric text features, giving more weight to patterns that help distinguish functions.
2. **Character n-grams** preserve local syntax and operator patterns that word tokenization might discard.
3. **Logistic Regression** learns feature weights for binary classification: CLEAN or BUGGY.
4. **Splitting by `source_id`** keeps an original function and its mutation together, preventing leakage of nearly identical code between splits.
5. **Easiest/hardest mutation type** is determined below by the highest/lowest test recall. This describes this dataset, not all Python bugs.

The labels are synthetic: CLEAN means the original function, and BUGGY means an operator was changed. These labels do not prove whether the function behaves correctly.

In [7]:
easiest = mutation_results["recall"].idxmax()
hardest = mutation_results["recall"].idxmin()
print(f"Easiest: {easiest} (recall {mutation_results.loc[easiest, 'recall']:.2%})")
print(f"Hardest: {hardest} (recall {mutation_results.loc[hardest, 'recall']:.2%})")

Easiest: comparison_operator (recall 74.28%)
Hardest: boolean_operator (recall 54.93%)
